# 06 – Deployment prototype: database, saved model, batch scoring and API
**QM640 Data Analytics Capstone – Binesh Balakrishnan**

This notebook runs the deployment prototype end to end:
1. Build the SQLite database `data/db/itsm.sqlite` from the CSV files (`src/build_database.py`).
2. Train and save the deployable model `models/sla_breach_rf.joblib` (`src/train_model.py`).
3. Replay May 2016 as "new" tickets and write risk scores to the database (`src/score_batch.py`).
4. Query the scored queue with SQL, as the dashboard does.
5. Call the REST scoring service in-process (`app/api.py`).

The dashboard itself (`app/dashboard.py`) is a Streamlit app; run it on a laptop with `streamlit run app/dashboard.py`.
The database and the model file are rebuilt by this notebook and are not stored in GitHub (see `.gitignore`).

In [1]:
import os, subprocess
repo = '/content/QM640_data_analytics_capstone'
if os.path.exists('/content') and not os.path.exists(repo):   # Colab: clone the repository
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/binesh29/QM640_data_analytics_capstone.git', repo], check=True)
if os.path.exists(repo):
    os.chdir(os.path.join(repo, 'notebooks'))
os.makedirs('../figures', exist_ok=True); os.makedirs('../results', exist_ok=True)
print(os.getcwd())

/home/claude/work/repo/notebooks


In [2]:
import sys, json, sqlite3
import pandas as pd
sys.path.insert(0, '../src'); sys.path.insert(0, '..')
import build_database, train_model, score_batch, sla_model

## 1. Build the database

In [3]:
print(build_database.build())

{'event_log': 141712, 'incidents': 24918, 'risk_scores': 0}


## 2. Train and save the deployable model
Random forest (notebook 04 hyperparameters) fitted on the first 80% of the training period, Platt recalibration on the last 20%,
risk bands from the recalibration window (High = top 20% of scores, Medium = next 20%). The hold-out block is the same test set as notebook 04.

In [4]:
meta = train_model.train()
print(json.dumps(meta['holdout'], indent=1)); print(meta['calibration_window'], meta['thresholds_raw'])

{
 "n": 7476,
 "auc": 0.776,
 "mean_probability": 0.247,
 "observed_rate": 0.233,
 "share_high": 0.222,
 "breach_rate_by_band": {
  "High": 0.504,
  "Low": 0.102,
  "Medium": 0.297
 }
}
['2016-04-18 11:05:00', '2016-05-02 11:26:00'] {'high': 0.5789918425482411, 'medium': 0.4646891627136427}


## 3. Batch-score May 2016 and store the scores

In [5]:
res = score_batch.score_window('2016-05-02', '2016-06-01')
print(len(res), 'incidents scored'); res['risk_band'].value_counts()

7428 incidents scored


risk_band
Low       4312
Medium    1568
High      1548
Name: count, dtype: int64

## 4. Query the scored queue (what the dashboard shows)

In [6]:
con = sqlite3.connect(sla_model.DB_PATH)
by_band = pd.read_sql('''SELECT risk_band, COUNT(*) AS incidents, ROUND(AVG(breach_probability), 3) AS mean_probability,
                                ROUND(AVG(actual_breach), 3) AS observed_breach_rate
                         FROM v_open_queue GROUP BY risk_band ORDER BY mean_probability DESC''', con)
day = pd.read_sql("SELECT * FROM v_open_queue WHERE substr(opened_at, 1, 10) = '2016-05-03'", con)
use_case = {'day': '2016-05-03', 'tickets': len(day), 'high': int((day['risk_band'] == 'High').sum()),
            'breaches': int(day['actual_breach'].sum()), 'breaches_in_high': int(day.loc[day['risk_band'] == 'High', 'actual_breach'].sum()),
            'expected_breaches': round(float(day['breach_probability'].sum()), 1)}
print(use_case); con.close(); by_band

{'day': '2016-05-03', 'tickets': 353, 'high': 70, 'breaches': 76, 'breaches_in_high': 41, 'expected_breaches': 83.6}


,risk_band,incidents,mean_probability,observed_breach_rate
0,High,1548,0.534,0.500
1,Medium,1568,0.294,0.302
2,Low,4312,0.114,0.099


## 5. Call the REST scoring service in-process

In [7]:
import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'httpx'], check=False)
from fastapi.testclient import TestClient
from app.api import app
client = TestClient(app)
print(client.get('/health').json())
ticket = {'number': 'INC_DEMO', 'opened_at': '2016-05-03 09:15:00', 'log_priority': '3 - Moderate', 'log_impact': '2 - Medium',
          'log_urgency': '2 - Medium', 'log_assignment_group': 'Group 70', 'log_category': 'Category 26', 'log_contact_type': 'Phone'}
print(client.post('/score', json=[ticket]).json())

{'status': 'ok', 'model_version': 'rf-v1.0'}
[{'number': 'INC_DEMO', 'breach_probability': 0.303, 'risk_score_raw': 0.528, 'risk_band': 'Medium', 'suggested_action': 'Watch: check at next queue review'}]


In [8]:
json.dump({'model_card': meta, 'score_bands_may2016': by_band.to_dict('records'), 'use_case': use_case},
          open('../results/deployment.json', 'w'), indent=1, default=str); print('saved')

saved
